# Task 3 : Fitting a two-compartment (bi-exponential) T2 model

Constrained bi-exponential fitting, initialisation sensitivity, and bootstrap CIs across the preterm cohort.

In [ ]:
import sys, time, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from utils import load_preterm_cohort, load_adult_case, mean_decay_curve
from models import bi_exp_fixed, fit_bi_fixed, fit_bi_free
from analysis import fit_cohort_biexp_fixed, bootstrap_cohort_biexp
from plotting import plot_biexp_decomposition, plot_ci_intervals, plot_ci_boxplots

## 3.1 : Load cohort

In [ ]:
preterm, _ = load_preterm_cohort('../data', exclude_file='../data/preterm_exclusions.csv')
case2 = load_adult_case(2)
print(f"Cohort: {len(preterm)} subjects, case 2: {len(case2['TE'])} echoes")

## 3.2 : Initialisation sensitivity

In [ ]:
wm2 = mean_decay_curve(case2, 3)
v_inits = [0.05, 0.10, 0.20, 0.40]

print("Free-T2 fits on Case 2 WM (22 echoes):")
for v0 in v_inits:
    S0, v, T2s, T2l = fit_bi_free(case2['TE'], wm2, v_init=v0)
    print(f"  v_init={v0:.2f}: v={v:.3f}, T2s={T2s:.1f}, T2l={T2l:.1f}")

print("\nFixed-T2 fits : stable across all initialisations:")
for v0 in v_inits:
    S0, v = fit_bi_fixed(case2['TE'], wm2, T2s=20, T2l=80, v_init=v0)
    print(f"  v_init={v0:.2f}: v={v:.3f}")

## 3.3 : Cohort fixed-T2 fits

In [ ]:
df_fixed = fit_cohort_biexp_fixed(preterm)
print("Cohort means per tissue:")
print(df_fixed.groupby('tissue')['v'].agg(['mean', 'std', 'count']).round(3).to_string())

## 3.4 : Bi-exponential decomposition

In [ ]:
ds = preterm[0]
S0, v = fit_bi_fixed(ds['TE'], mean_decay_curve(ds, 3), T2s=20, T2l=80)
plot_biexp_decomposition(ds, S0, v)

## 3.5 : Bootstrap confidence intervals

In [ ]:
t0 = time.time()
df_ci = bootstrap_cohort_biexp(preterm, n_boot=200)
print(f"Bootstrap finished in {time.time()-t0:.1f} s")
print("\nCohort summary:")
print(df_ci.groupby('tissue').agg(
    mean_v=('v', 'mean'), SD_v=('v', 'std'),
    median_CI=('CI_width', 'median')).round(3).to_string())

plot_ci_intervals(df_ci)
plot_ci_boxplots(df_ci)